In [1]:
# install deps
!pip install -q sherpa-onnx
!pip install -q soundfile
!pip install -q --upgrade huggingface_hub


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.


In [2]:
import os
import sherpa_onnx
import soundfile as sf
from huggingface_hub import snapshot_download

In [13]:
def print_trans():
  file = "eng-test-wav/trans.txt"
  if not os.path.exists(file):
    print(f"Please provide a valid text file at: {file}")
    return

  with open(file, "r") as f:
    print(f"{f.read()}")


In [14]:
print_trans()

Please provide a valid text file at: eng-test-wav/trans.txt


In [4]:
def split_wav(recognizer, audio, sample_rate):
  chunk_duration = 5 # 3 # 10 # seconds
  samples_per_chunk = sample_rate * chunk_duration
  total_samples = len(audio)

  # loop and process 10-sec chunks
  for i in range(0, total_samples, samples_per_chunk):
    chunk_audio = audio[i : i + samples_per_chunk]
    # create stream and decode for this chunk
    stream = recognizer.create_stream()
    stream.accept_waveform(sample_rate, chunk_audio)
    recognizer.decode_stream(stream)
    #print("text: ")
    print(stream.result.text)


In [15]:

def run_inference():
#def main():
  # 1. Download model files from Hugging Face
  model_dir="parakeet-en-model"
  # Check available files (model.onnx or model.int8.onnx and tokens.txt)
  onnx_model = os.path.join(model_dir, "model.int8.onnx")
  if not os.path.exists(onnx_model):
    onnx_model = os.path.join(model_dir, "model.onnx")
  tokens_path = os.path.join(model_dir, "tokens.txt")

  # 2. Create OfflineRecognizer for NeMo CTC
  recognizer = sherpa_onnx.OfflineRecognizer.from_nemo_ctc(
      model=onnx_model,
      tokens=tokens_path,
      num_threads=2,
      debug=False,
      provider="cpu",
  )
  # audio file path
  #audio_path = "test-eng-wav/0.wav"
  #audio_path = "test-eng-wav/1.wav"
  audio_path = "test-eng-wav/Obama.wav"
  #audio_path = "test-eng-wav/test01_20s.wav"
  # 3. Load an audio file (must be mono, 16 kHz typically, depending on the model)
  if not os.path.exists(audio_path):
    print(f"Please provide a valid audio file at: {audio_path}")
    return

  print("audio file:", audio_path)
  audio, sample_rate = sf.read(audio_path, dtype="float32")
  if audio.ndim > 1:
    audio = audio[:, 0]  # Take first channel if stereo

  # 4. Create stream and decode
  split_wav(recognizer, audio, sample_rate)

  print("------------ verify -------------")
  print_trans()


In [6]:
pip -q install psutil

In [16]:
import time
import os
import psutil
import tracemalloc
import threading

class MemoryTracker:
    def __init__(self, interval_seconds=0.1):
        self.interval = interval_seconds
        self.process = psutil.Process(os.getpid())
        self.peak_rss = 0
        self.peak_vms = 0
        self.is_running = False
        self._thread = None
        self.start_time = 0
        self.end_time = 0

    def _monitor(self):
        while self.is_running:
            try:
                mem_info = self.process.memory_info()
                # RSS (Resident Set Size): Actual physical RAM used by the process
                # VMS (Virtual Memory Size): Total virtual memory allocated
                if mem_info.rss > self.peak_rss:
                    self.peak_rss = mem_info.rss
                if mem_info.vms > self.peak_vms:
                    self.peak_vms = mem_info.vms
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                break
            time.sleep(self.interval)

    def start(self):
        """Starts monitoring memory usage in a background thread."""
        print("🚀 [MemoryTracker] Started monitoring...")
        tracemalloc.start()
        self.start_time = time.perf_counter()
        print("start time: ", self.start_time)
        self.is_running = True
        self._thread = threading.Thread(target=self._monitor, daemon=True)
        self._thread.start()

    def stop(self):
        """Stops monitoring and prints out the peak memory usage statistics."""
        self.is_running = False
        self.end_time = time.perf_counter()
        print("end time: ", self.end_time)
        if self._thread:
            self._thread.join()

        # Get internal python allocations
        current_alloc, peak_alloc = tracemalloc.get_traced_memory()
        tracemalloc.stop()

        print("\n📊 --- Memory Usage Report ---")
        print(f"🔹 Peak Physical RAM (RSS):  {self.peak_rss / (1024**2):.2f} MB")
        print(f"🔹 Peak Virtual Memory (VMS): {self.peak_vms / (1024**2):.2f} MB")
        print(f"🔹 Peak Python Objects Alloc: {peak_alloc / (1024**2):.2f} MB")
        print(f"🔹 Execution Time : {self.end_time - self.start_time:.2f} seconds")
        print("-------------------------------\n")


if __name__ == "__main__":
    # Import the function from your specific model script
    # from my_model import run_inference
    # from test_hf_eng_parakeet import run_inference

    tracker = MemoryTracker(interval_seconds=0.1)
    print("start tracking...")
    tracker.start()

    print("🤖 Running model...")
    # Call your imported function here
    run_inference()

    tracker.stop()
    print("stopped tracking...")

start tracking...
🚀 [MemoryTracker] Started monitoring...
start time:  1557.872077724
🤖 Running model...
audio file: test-eng-wav/Obama.wav

Thank you. Thank you. Thank you, everybody right
All right, everybody go ahead and have a seat. How's everybody doing today
How about Tim Spicer

I am here with students at Wakefield High School
in Arlington, Virginia. and we've got students tuning in
all across America from kindergarten through twelfth grade
And I am just so glad that all could join us today and I want to thank Wakefield
for being such an outstanding host. Give yourselves a big honor try

know that for many of you, today is the first day of school. for those
For those of you in kindergarten or starting middle or high school, it's your first day
in a new school, so it's understandable if you're a little nervous
I imagine there's some seniors out there who are feeling pretty good right now.
one more year to go. And no matter what grade you
some of you are probably wishing it was st